In [9]:
!pip install kneed

In [11]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
import scipy.io
import pandas as pd
from kneed import KneeLocator
import pandas as pd
from sklearn.metrics import roc_auc_score
import h5py

# # #Open the v7.3 .mat file using h5py
# with h5py.File('http.mat', 'r') as f:
#     # Assuming 'X' is the dataset you want
#     data = f['X'][:]  # Load the data from the dataset
#     datas = np.array(data).T  # Transpose if necessary to match original shape
#     # Load the labels (y)
#     datay = np.array(f['y']).T  # Transpose if needed
#     y = datay.flatten().astype(int)  # Make sure y is a 1D array

dataR = scipy.io.loadmat('glass.mat')
dataX = dataR['X']
datas = np.array(dataX)
datay = dataR['y']
y = np.array(datay)

# df_X = pd.read_csv('/content/glass.csv', header=None)
# df_y = pd.read_csv('/content/glassLabels.csv', header=None)
# datas = np.array(df_X)
# y = np.array(df_y)

#Initialize an empty array to store the first 10 data points
buffer = np.empty((0, datas.shape[1]))  # Adjust the number of columns accordingly
weights = np.array([])  # Weight buffer
limit = 100  # Maximum size of the buffer
g = 2

# Initialize an array to record LOF scores for each index
lof_scores = np.full(datas.shape[0], np.nan)  # Use NaN as placeholder

#To keep track of original indices in the buffer
indices_in_buffer = []

drawValve = 0

#Simulate reading data one by one
for i, row in enumerate(datas):
  #Append the new data
  buffer = np.vstack([buffer, row.reshape(1, -1)])
  indices_in_buffer.append(i)

  if weights.size > 0:
    weights *= 0.999  # Reduce all weights by 0.1%

  weights = np.append(weights, 1.0)

  if buffer.shape[0] >= limit:

    [L,W] = buffer.shape

    #Compute squared Euclidean pairwise distances
    dist0 = pdist(buffer, metric='euclidean') ** 2

    # Convert (dist0) to a square distance matrix
    dist1 = squareform(dist0)

    # Compute the median of distances
    median_dist = np.median(dist0)

    # Select only distance that are less than median_dist
    filtered_deviations = dist0[dist0 < median_dist]

    # Compute the second median
    mad2 = np.median(filtered_deviations)

    # Define threshold (e.g., 50% of the median)
    threshold = 0.5 * median_dist

    # Check tightness or sparsity
    if threshold < mad2:
        g = 3
    else:
        g = 2

    averdist = np.mean(dist0)

    # Loop to update dist0 and averdist for G iterations
    for ii in range(g):
      # Remove distances greater than averdist
      dist0 = dist0[dist0 <= averdist]

      # Recompute the mean of the remaining distances
      averdist = np.mean(dist0)

    #Create the identity matrix of size L
    eye_matrix = np.eye(L)

    #Calculate the maximum value in dist1
    max_dist = np.max(dist1)

    #Modify the distance matrix to exclude self-distances by adding a large number to the diagonal
    dist2 = dist1 + eye_matrix * (max_dist + 1)

    #Initialize the matrix that will represent affinity graph with zeros
    A = np.zeros((L, L))

    #Set A entries to 1 where distances are less than or equal to the average distance
    A[dist2 <= averdist] = 1

    #Compute the average number of neighbors (NN)
    NN = int(np.ceil(np.mean(np.sum(A, axis=1))))

    #Sort each row in ascending order and get the indices of the sorted values
    A2 = np.argsort(dist2, axis=1)

    #Extract nearest neighbors and get the corresponding values
    nn = A2[:, :NN]

    # Initialize reachability distance matrix
    RD = np.zeros((L, NN))

    # Compute reachability distances
    for i in range(L):
      for j_idx, j in enumerate(nn[i]):
        # reachability distance = max(k-distance of neighbor, distance between i and neighbor)
        k_distance_j = dist2[j, nn[j, NN-1]]  # k-distance of neighbor j
        RD[i, j_idx] = max(k_distance_j, dist2[i, j])

    # Local reachability density
    LRD = 1 / (np.mean(RD, axis=1))

    # LOF computation
    LOF = np.zeros(L)
    for i in range(L):
      LOF[i] = np.sum(LRD[nn[i]] / LRD[i]) / NN

    # Update the lof_scores array for the current indices in the buffer
    for idx, lof_value in zip(indices_in_buffer, LOF):
      lof_scores[idx] = lof_value

    num_to_remove = 1
    top_outlier_indices = np.argsort(LOF)[-num_to_remove:]
    removal_index = top_outlier_indices[0]

    sorted_lof = np.sort(LOF)  # Sort LOF scores
    lof_indices = np.arange(len(sorted_lof))  # Index positions

    #Subtract each LOF score from all others
    diff_matrix = LOF[:, None] - LOF

    #Remove self-pairs (diagonal)
    diff_matrix_no_self = diff_matrix[~np.eye(len(LOF), dtype=bool)].reshape(len(LOF), -1)

    #Calculate median differences
    median_differences = np.median(diff_matrix_no_self, axis=1)

    sort_m = np.sort(median_differences)
    sort_m_indices = np.arange(len(median_differences))

    knee_locator = KneeLocator(sort_m_indices, sort_m, curve="convex", direction="increasing", S=5)
    lof_threshold = sort_m[knee_locator.knee]  # Adaptive threshold

    num_to_remove = 1
    top_outlier_indices = np.argsort(median_differences)[-num_to_remove:]
    removal_index = top_outlier_indices[0]

    if median_differences[removal_index] > np.max(lof_threshold):  # Remove outlier if LOF is high

      buffer = np.delete(buffer, removal_index, axis=0)
      weights = np.delete(weights, removal_index)
      del indices_in_buffer[removal_index]
    else:
      removal_index = np.argmin(weights)  # Find the lowest-weighted data point

      buffer = np.delete(buffer, removal_index, axis=0)
      weights = np.delete(weights, removal_index)
      del indices_in_buffer[removal_index]

auc = roc_auc_score(y, lof_scores)
print(auc)

0.8319783197831978
